# 🔄 Основни работни потоци на агенти с Microsoft Foundry (Python)

## 📋 Урок за оркестрация на работни потоци

Този бележник представя мощните възможности на **Workflow Builder** в Microsoft Agent Framework. Научете как да създавате сложни, многостъпкови работни потоци на агенти, които могат безпроблемно да управляват комплексни бизнес процеси и да координират множество AI операции.

> **Бележка за миграция:** Този пример по-рано се позоваваше на GitHub Models. GitHub Models е остарял (ще бъде прекратен през юли 2026), затова сега използва **Microsoft Foundry** чрез `FoundryChatClient`, който таргетира Azure OpenAI **Responses API**.

## 🎯 Учебни цели

### 🏗️ **Архитектура на работния поток**
- **Workflow Builder**: Проектиране и оркестрация на сложни многостъпкови процеси
- **Събитийно задействано изпълнение**: Обработка на събития и преходи на състояния в работния поток
- **Визуален дизайн на работния поток**: Създаване и визуализация на структури на работни потоци
- **Интеграция с Microsoft Foundry**: Използване на AI модели в контекста на работни потоци

### 🔄 **Оркестрация на процеси**
- **Последователни операции**: Свързване на множество задачи на агенти в логичен ред
- **Условна логика**: Имплементиране на точки на вземане на решения и разклоняващи се работни потоци
- **Обработка на грешки**: Здрава възстановимост при грешки и устойчивост на работния поток
- **Управление на състояния**: Проследяване и управление на състоянието на изпълнение на работния поток

### 📊 **Модели на работни потоци за предприятия**
- **Автоматизация на бизнес процеси**: Автоматизиране на сложни организационни работни потоци
- **Координация на множество агенти**: Координиране на множество специализирани агенти
- **Мащабируемо изпълнение**: Проектиране на работни потоци за операции в предприятия от мащабен тип
- **Мониторинг и наблюдаемост**: Проследяване на производителността и резултатите от работния поток

## ⚙️ Предварителни изисквания и настройка

### 📦 **Задължителни зависимости**

Инсталирайте Agent Framework с възможности за работни потоци:

```bash
pip install agent-framework -U
```

### 🔑 **Конфигурация на Microsoft Foundry**

Влезте с Azure CLI (`az login`), за да може `AzureCliCredential` да се удостоверява, след което задайте детайлите на вашия Microsoft Foundry проект.

**Настройка на околната среда (.env файл):**
```env
AZURE_AI_PROJECT_ENDPOINT=https://<your-project>.services.ai.azure.com
AZURE_AI_MODEL_DEPLOYMENT_NAME=gpt-4.1-mini
```

### 🏢 **Бизнес случаи за предприятия**

**Примери за бизнес процеси:**
- **Въвеждане на клиенти**: Многостъпкови работни потоци за проверка и настройка
- **Поток за съдържание**: Автоматично създаване, преглед и публикуване на съдържание
- **Обработка на данни**: ETL работни потоци с AI захранвана трансформация
- **Осигуряване на качество**: Автоматизирани тестове и процеси за валидиране

**Ползи от работни потоци:**
- 🎯 **Надеждност**: Детерминистично изпълнение с възстановяване при грешки
- 📈 **Мащабируемост**: Управление на автоматизация при голям обем процеси
- 🔍 **Наблюдаемост**: Пълни одитни следи и мониторинг
- 🔧 **Поддръжка**: Визуален дизайн и модулни компоненти

## 🎨 Шаблони за дизайн на работни потоци

### Основна структура на работния поток
```mermaid
graph TD
    A[Старт] --> B[Задача на агента 1]
    B --> C{Точка на решение}
    C -->|Успех| D[Задача на агента 2]
    C -->|Неуспех| E[Обработчик на грешки]
    D --> F[Край]
    E --> F
```

**Основни компоненти:**
- **WorkflowBuilder**: Основен двигател за оркестрация
- **WorkflowEvent**: Обработка на събития и комуникация
- **WorkflowViz**: Визуално представяне и отстраняване на грешки на работния поток

Нека създадем първия ви интелигентен работен поток! 🚀


In [ ]:
# Already covered by repo-level requirements.txt; left for reference.
# !pip install agent-framework -U

In [ ]:
# Core components for building sophisticated agent workflows
from agent_framework import WorkflowBuilder, WorkflowEvent, WorkflowViz
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential


In [ ]:
# 📦 Import Environment and System Utilities
# Essential libraries for configuration and environment management

import os                      # 🔧 Environment variable access
from dotenv import load_dotenv # 📁 Secure configuration loading

In [ ]:
# 🔧 Initialize Environment Configuration
# Load Microsoft Foundry project settings from .env file
load_dotenv()


In [ ]:
# Configure the Microsoft Foundry client with keyless authentication.
# FoundryChatClient targets the Azure OpenAI Responses API.
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)


In [ ]:
REVIEWER_NAME = "Concierge"
REVIEWER_INSTRUCTIONS = """
    You are an are hotel concierge who has opinions about providing the most local and authentic experiences for travelers.
    The goal is to determine if the front desk travel agent has recommended the best non-touristy experience for a traveler.
    If so, state that it is approved.
    If not, provide insight on how to refine the recommendation without using a specific example. 
    """

In [ ]:
FRONTDESK_NAME = "FrontDesk"
FRONTDESK_INSTRUCTIONS = """
    You are a Front Desk Travel Agent with ten years of experience and are known for brevity as you deal with many customers.
    The goal is to provide the best activities and locations for a traveler to visit.
    Only provide a single recommendation per response.
    You're laser focused on the goal at hand.
    Don't waste time with chit chat.
    Consider suggestions when refining an idea.
    """

In [ ]:
reviewer_agent = provider.as_agent(
    name=REVIEWER_NAME,
    instructions=REVIEWER_INSTRUCTIONS,
)

front_desk_agent = provider.as_agent(
    name=FRONTDESK_NAME,
    instructions=FRONTDESK_INSTRUCTIONS,
)


In [ ]:
workflow = (
    WorkflowBuilder(start_executor=front_desk_agent)
    .add_edge(front_desk_agent, reviewer_agent)
    .build()
)

In [ ]:
print("Generating workflow visualization...")
viz = WorkflowViz(workflow)
# Print out the mermaid string.
print("Mermaid string: \n=======")
print(viz.to_mermaid())
print("=======")
# Print out the DiGraph string.
print("DiGraph string: \n=======")
print(viz.to_digraph())
print("=======")
# SVG export needs the optional graphviz extra plus the graphviz system binary;
# fall back gracefully if it is not available.
try:
    svg_file = viz.export(format="svg")
    print(f"SVG file saved to: {svg_file}")
except ImportError as e:
    svg_file = None
    print(f"SVG export skipped (install graphviz to enable): {e}")

In [ ]:
class DatabaseEvent(WorkflowEvent): ...

In [ ]:
# Display the exported workflow SVG inline in the notebook

from IPython.display import SVG, display, HTML
import os

print(f"Attempting to display SVG file at: {svg_file}")

if svg_file and os.path.exists(svg_file):
    try:
        # Preferred: direct SVG rendering
        display(SVG(filename=svg_file))
    except Exception as e:
        print(f"⚠️ Direct SVG render failed: {e}. Falling back to raw HTML.")
        try:
            with open(svg_file, "r", encoding="utf-8") as f:
                svg_text = f.read()
            display(HTML(svg_text))
        except Exception as inner:
            print(f"❌ Fallback HTML render also failed: {inner}")
else:
    print("❌ SVG file not found. Ensure viz.export(format='svg') ran successfully.")


In [ ]:
# Workflow.run_stream is no longer part of the public API; the current Workflow
# returns a results object whose `get_outputs()` produces the AgentResponse from
# each output executor. The reviewer (last stage) is the only output here.
events = await workflow.run("I would like to go to Paris.")
outputs = events.get_outputs()
result = outputs[0].text if outputs else ""

In [ ]:
result.replace("None", "")

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Отказ от отговорност**:
Този документ е преведен с помощта на AI преводачески услуга [Co-op Translator](https://github.com/Azure/co-op-translator). Въпреки че се стремим към точност, моля имайте предвид, че автоматизираните преводи могат да съдържат грешки или неточности. Оригиналният документ на неговия роден език трябва да се счита за авторитетен източник. За критична информация се препоръчва професионален човешки превод. Ние не носим отговорност за каквито и да е недоразумения или неправилни тълкувания, произтичащи от използването на този превод.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
